# nébuleuses
$\rightarrow$ **calcul du rougissement, de la température et de la densité électronique**

# la cible : m97

|||
|---|---|
|OBJECT|	m97 |
|EXPTIME2|	7 x 600 s|
|DATE-OBS|	2026-02-24T20:14:21.969|
|BSS_SITE|	CALC|
|BSS_INST|	C11 + Dados200 + 25mic + PO_UranusM|
|||

 
à faire : 
- 20161006_ngc6543
- 20240709_tcrb_m57
- 20240710_chcyg_tcrb_m16_m57
- 

In [1]:
%matplotlib widget
import numpy as np
from spectra_widget import SpectraWidget

# 1. Afficher le dashboard
db = SpectraWidget()
db.show()


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum
from astropy import units as u

filename = 'data/plouis/_m97_20260224_843.fits'
#filename = '../../../CAPTURES/20260224_m97_merak_procyon/_m97_20260224_843.fits'

# Chargement
sp = Spectrum.read(filename, units=u.adu)

sp = Spectrum(
    spectral_axis=sp.spectral_axis,
    flux=sp.flux.value * u.Unit('adu'),
    meta=sp.meta
)


print('flux unit='+str(sp.flux.unit))
print('lambda unit='+str(sp.spectral_axis.unit))
sp.meta

flux unit=adu
lambda unit=Angstrom


{'header': SIMPLE  =                    T / conforms to FITS standard                      
 BITPIX  =                  -32 / array data type                                
 NAXIS   =                    1 / number of array dimensions                     
 NAXIS1  =                 4933                                                  
 CRVAL1  =    4300.205721228189                                                  
 CDELT1  =   0.5772947513942199                                                  
 EXPTIME =               4200.0                                                  
 OBJNAME = 'm97     '                                                            
 OBJECT  = 'm97     '                                                            
 EXPTIME2= '7 x 600 s'                                                           
 BSS_ITRP=                  500                                                  
 SPE_RPOW=                  500                                                  
 BSS_V

In [3]:
db.show_spectrum(sp.spectral_axis, sp.flux, label=sp.meta['header']['OBJECT'])

INFO: affichage du spectre 'm97' : 4933 pts, X:[4300.2:7147.4]


In [4]:
from specutils.fitting import fit_generic_continuum
from astropy.modeling import models
from scipy.signal import medfilt


# 1. On extrait le flux brut pour numpy
raw_flux = np.array(sp.flux.value, dtype=np.float64)

# 2. On calcule le filtre médian (le "plancher" de bruit)
# Le kernel_size doit être un nombre impair assez large (ex: 201 pixels)
# pour "enjamber" les raies et ne voir que le fond.
fond_estime = medfilt(raw_flux, kernel_size=355)

# On remet le fond à zéro
flux_net = (raw_flux - fond_estime) * sp.flux.unit

# 4. On crée le nouveau Spectre "Nettoyé"
spec_norm = Spectrum(spectral_axis=sp.spectral_axis, flux=flux_net, meta=sp.meta)

"""
continuum_model = models.Polynomial1D(degree=5)
fitted_continuum = fit_generic_continuum(sp, model=continuum_model)

# on retire le continuum
flux_norm = sp.flux - fitted_continuum(sp.spectral_axis)

# on crée le nouveau Spectre normalisé
spec_norm = Spectrum(spectral_axis=sp.spectral_axis, flux=flux_norm, meta=sp.meta)
"""

db.clear_spectra()
db.show_spectrum(spec_norm.spectral_axis, spec_norm.flux, label=spec_norm.meta['header']['OBJECT'] + ' sans cont')
db.ax_spec.axhline(0, color='red', linestyle=':', label='Zéro')


INFO: affichage du spectre 'm97 sans cont' : 4933 pts, X:[4300.2:7147.4]


In [5]:
from specutils.analysis import line_flux
from specutils import SpectralRegion
from specutils.manipulation import extract_region
from astropy import units as u
from scipy.integrate import trapezoid
from specutils.fitting import fit_lines

def fit_line_gauss(spec, center, left, right, sig, show_plot=True):
    """
    ajuste une raie avec un continuum à zéro
    retourne son flux (intégrale)
    """
    _reg = SpectralRegion(left, right) 
    sub_reg = extract_region(spec, _reg)
    ampl = np.max(sub_reg.flux)

    fitter = fitting.LevMarLSQFitter()
    gauss = models.Gaussian1D(amplitude=ampl, mean=center, stddev=sig)
    fitted_reg = fitter(gauss, sub_reg.spectral_axis, sub_reg.flux)
    flux_fit = fitted_reg.amplitude.value * fitted_reg.stddev.value * np.sqrt(2 * np.pi)

    if show_plot:
        plt.figure(figsize=(8, 4))
        plt.plot(sub_reg.spectral_axis, sub_reg.flux, 'b-', label='brut')
        plt.plot(sub_reg.spectral_axis, fitted_reg(sub_reg.spectral_axis), 'r--', label='fit')
        plt.axvline(center.value, color='green', linestyle=':', label=str(center.value))
        plt.legend()
        plt.xlabel('Longueur d’onde (Å)')
        plt.ylabel('Flux')
        plt.grid(True)
        plt.show()

    return flux_fit

# H --> le rougissement
h_alpha = 6562 * u.AA
h_beta = 4861 * u.AA

flux_h_alpha = fit_line_gauss(spec_norm, h_alpha, h_alpha - 14  * u.AA, h_alpha + 12 * u.AA, 0.8, False)
flux_h_beta = fit_line_gauss(spec_norm, h_beta, h_beta - 12 * u.AA, h_beta + 16 * u.AA, 1.0, False)
print(f"Rapport des flux H fitted : {flux_h_alpha / flux_h_beta}")


# OIII --> la température
oiii_5007 = 5007 * u.AA
oiii_4959 = 4959 * u.AA
oiii_4363 = 4363 * u.AA

flux_oiii_5007 = fit_line_gauss(spec_norm, oiii_5007, oiii_5007 - 14  * u.AA, oiii_5007 + 16 * u.AA, 0.8, False)
flux_oiii_4959 = fit_line_gauss(spec_norm, oiii_4959, oiii_4959 - 14  * u.AA, oiii_4959 + 16 * u.AA, 0.8, False)
flux_oiii_4363 = fit_line_gauss(spec_norm, oiii_4363, oiii_4363 - 10  * u.AA, oiii_4363 + 10 * u.AA, 0.8, False)

print (f"{flux_oiii_5007=:.2f}")
print (f"{flux_oiii_4959=:.2f}")
print (f"{flux_oiii_4363=:.2f}")

# SII --> pas utilisable
sii_6717 = 6717 * u.AA
sii_6731 = 6731 * u.AA
flux_sii_6717 = fit_line_gauss(spec_norm, sii_6717, sii_6717 - 16  * u.AA, sii_6717 + 16 * u.AA, 0.8, False)
flux_sii_6731 = fit_line_gauss(spec_norm, sii_6731, sii_6731 - 10  * u.AA, sii_6731 + 10 * u.AA, 0.8, False)
print (f"{flux_sii_6717=:.2f}")
print (f"{flux_sii_6731=:.2f}")

print(f"Rapport des flux SII fitted : {flux_sii_6717 / flux_sii_6731}")



Rapport des flux H fitted : 2.9250587904612355
flux_oiii_5007=205.06
flux_oiii_4959=65.48
flux_oiii_4363=7.36
flux_sii_6717=4.51
flux_sii_6731=-1.30
Rapport des flux SII fitted : -3.481441450556211


- Avec ce rapport, on peut calculer l'extinction (logarithmique) : Cardelli (CCM 89)
- La poussière "rougit" le spectre en absorbant davantage le bleu ($H\beta$) que le rouge ($H\alpha$).
- La formule standard est :$$c(H\beta) = \frac{1}{K_{H\alpha} - K_{H\beta}} \cdot \log_{10} \left( \frac{R_{mesuré}}{R_{théorique}} \right)$$



In [6]:
def calculer_extinction(ha_flux, hb_flux, r_theo=2.85):
    """
    Calcule le coefficient d'extinction c(Hb) 
    Basé sur le décrément de Balmer Ha/Hb.
    """
    # 1. Calcul du rapport observé
    r_obs = ha_flux / hb_flux
    
    # 2. Paramètres de la loi de Cardelli (CCM 89)
    # f(lambda) pour Ha (6563A) relatif à Hb (4861A)
    # Selon la loi standard, l'écart de pente (f_Ha - f_Hb) est d'environ -0.334
    delta_f = -0.334 
    
    # 3. Formule de calcul de c(Hb)
    # c(Hb) = log10(R_obs / R_theo) / [f(Hb) - f(Ha)]
    c_hb = np.log10(r_obs / r_theo) / (0 - delta_f)    # 0 -> f(Hb) 
    
    return c_hb, r_obs

c_val, rapport = calculer_extinction(flux_h_alpha, flux_h_beta)

print(f"Rapport mesuré Ha/Hb : {rapport:.3f}")
print(f"Coefficient d'extinction c(Hb) : {c_val:.3f}")

# --- Calcul du rougissement E(B-V) ---
# Rapport constant : c(Hb) = 1.45 * E(B-V) environ
e_bv = c_val / 1.45
print(f"Rougissement E(B-V) : {e_bv:.3f} mag")



Rapport mesuré Ha/Hb : 2.925
Coefficient d'extinction c(Hb) : 0.034
Rougissement E(B-V) : 0.023 mag


In [7]:
def analyse_temperature_oiii(f4363, f4959, f5007, c_hb=0.034):
    """
    Calcule le ratio de contrôle, applique le dérougissement 
    et estime la température électronique Te.
    """
    # 1. Ratio de contrôle (Théorique ~ 2.98)
    ratio_controle = f5007 / f4959
    
    # 2. Dérougissement des flux (Loi de Cardelli simplifiée)
    # f_lambda est l'écart relatif à Hb (4861A)
    f_4363 = 0.13   # Bleu : plus d'extinction
    f_4959 = -0.02  # Vert
    f_5007 = -0.03  # Vert : moins d'extinction
    
    # Flux intrinsèques (I = F * 10^(c_hb * f_lambda))
    i4363 = f4363 * (10**(c_hb * f_4363))
    i4959 = f4959 * (10**(c_hb * f_4959))
    i5007 = f5007 * (10**(c_hb * f_5007))
    
    # 3. Calcul du ratio de température R_OIII
    # Formule : (I(4959) + I(5007)) / I(4363)
    r_oiii = (i4959 + i5007) / i4363
    
    # 4. Estimation de la Température Te (Formule de Liu et al. 2000)
    # Valable pour les densités usuelles des NP
    te = 14320 / np.log(0.13 * r_oiii)
    
    return {
        "ratio_5007_4959": ratio_controle,
        "r_oiii_derougi": r_oiii,
        "te_kelvin": te
    }

resultats = analyse_temperature_oiii(flux_oiii_4363, flux_oiii_4959, flux_oiii_5007, c_val)

print(f"--- Résultats M97 ---")
print(f"Ratio de contrôle (5007/4959) : {resultats['ratio_5007_4959']:.3f} (Attendu: ~2.98)")
print(f"Ratio de température R_OIII : {resultats['r_oiii_derougi']:.2f}")
print(f"Température Électronique Te : {resultats['te_kelvin']:.0f} K")

--- Résultats M97 ---
Ratio de contrôle (5007/4959) : 3.132 (Attendu: ~2.98)
Ratio de température R_OIII : 36.30
Température Électronique Te : 9229 K


Cardelli, Clayton et Mathis (1989) est la "Bible" de l'extinction. 
- Elle donne une formule complexe qui dépend d'un paramètre appelé $R_V$ (le rapport entre l'extinction visuelle et le rougissement).
- Dans le milieu interstellaire standard, on fixe $R_V = 3.1$.
- La formule complète de Cardelli utilise des polynômes de degré 7 ou 8 pour coller parfaitement à la courbe d'absorption de l'UV jusqu'à l'Infra-rouge.

Le "Rapport de l'Oxygène" (Ratio 5007 / 4959)

- Le ratio mesuré : $205.06 / 65.48 = \mathbf{3.13}$.
- En physique atomique, ce ratio est fixé par la nature à 2.98 (souvent arrondi à 3.0).
- Calcul de la Température Électronique ($T_e$)
  1. Avec la raie  $[O III] \lambda 4363$ fiable mais isolée, on peut estimer la chaleur du gaz.
  2. Le  ratio de température ($R_{OIII}$) :$$R_{OIII} = \frac{Flux(5007) + Flux(4959)}{Flux(4363)} = \approx \mathbf{36.30}$$
  3. Pour transformer ce chiffre en degrés Kelvin, on utilise une formule simplifiée (Formule de Liu et al. 2000, valable pour les densités typiques des nébuleuses planétaires) :
   $$T_e \approx \frac{14320}{\ln(0.13 \times R_{OIII})}$$
  5. On obtient : $T_e \approx \frac{14320}{\ln(4.78)} \approx \frac{14320}{1.56} \approx \mathbf{9\,200\text{ K}}$
  7. Les publications scientifiques situent le Hibou entre 9 000 K et 10 500 K.
  9. Le rapport $H\alpha/H\beta$ de 2.925  se compare au rapport théorique de 2.85 --> $c(H\beta)$ = 0.034 signifie que le ciel était bien transparent.

En résumé : 

| **Paramètre** | **Valeur mesurée** | **Valeur pro** |
|---|---|--- |
| **Rapport Balmer ($H\\alpha/H\\beta$)** | **2.92** | 2.85 |
| **Ratio $[O III]$ (5007/4959)** | **3.13** | 2.98 |
| **Température ($T_e$)** | **~9 200 K** | 9 000 - 10 500 K |